In [1]:
import os
import shutil
import sys
import subprocess

In [2]:
# Dataset Path
dataset_path = "../data/20250123_Sideview_40Days"

# Clean the dataset directory
if os.path.exists(dataset_path):
    shutil.rmtree(dataset_path)

# Create the dataset directory
if not os.path.exists(dataset_path):
    os.makedirs(dataset_path)

# Create images directory
images_path = dataset_path + "/images"
if not os.path.exists(images_path):
    os.makedirs(images_path)

# Create xml files directory
xml_path = dataset_path + "/xml"
if not os.path.exists(xml_path):
    os.makedirs(xml_path)

output_path = dataset_path
# Get absolute path
output_path = os.path.abspath(output_path)

In [3]:
# from tqdm import tqdm
# program_path = "../src/GenerateDataset/build_release"

# n_iter = 10000

# # Remove and recreate output folder
# shutil.rmtree(output_path, ignore_errors=True)
# os.makedirs(output_path, exist_ok=True)


# # Check the current platform
# import platform
# current_platform = platform.system()
# # Set the accelerator based on the platform
# if current_platform == "Darwin":
#     pass
# else:
#     os.environ["DISPLAY"] = ":12.0"
    
# # Generate dataset
# for i in tqdm(range(n_iter)):
#     # print("Generating image: ", i)
#     seed = i
#     image_name = f"cowpea_{i:04d}" 
#     # Generate image 
#     # Construct the command
#     command = ""
#     command += f"cd {program_path} && ./main " 
#     command += f"-h 1.0 -o {output_path} -seed {seed} -name {image_name} -xml -tile none -g"
#     result = subprocess.run(command, shell=True, capture_output=True, text=True)


#     # Check if the command was successful
#     if result.returncode == 0:
#         # print("Command executed successfully")
#         # print(result.stdout)  # Print the standard output
#         pass
#     else:
#         print(result.stdout)  # Print the standard output
#         print(result.stderr)  # Print the error output
#         raise("Command failed")
#         pass

import os
import subprocess
import concurrent.futures
from tqdm import tqdm

import platform
current_platform = platform.system()

# from tqdm import tqdm
program_path = "../src/GenerateDataset/build"

n_iter = 1000

if current_platform == "Darwin":
    pass
else:
    os.environ["DISPLAY"] = ":12.0"


# Function to re-render a single XML file
def generate_xml(image_name, seed, rotation=False):
    # Generate image 
    # Construct the command
    command = ""
    command += f"cd {program_path} && ./main " 
    command += f"-h 1.0 -o {output_path} -name {image_name} -seed {seed} -tile none -g -xml -days 40"
    if rotation:
        command += " -r"
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    return result

# Re-render XML files in parallel with progress bar
with concurrent.futures.ThreadPoolExecutor() as executor:
    xml_list = [f"cowpea_{i:04d}" for i in range(n_iter)]
    futures = [executor.submit(generate_xml, filename, i) for i, filename in enumerate(xml_list)]
    
    # Use tqdm to show progress bar
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Re-rendering XML files"):
        result = future.result()
        # Check if the command was successful
        if result.returncode != 0:
            print(result.stdout)  # Print the standard output
            print(result.stderr)  # Print the error output
            raise Exception("Command failed")

Re-rendering XML files: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [24:23<00:00,  1.46s/it]


In [4]:
import os
import subprocess
import concurrent.futures
from tqdm import tqdm

import platform
current_platform = platform.system()

# from tqdm import tqdm
program_path = "../src/GenerateDataset/build"
if current_platform == "Darwin":
    pass
else:
    os.environ["DISPLAY"] = ":12.0"
    
# Function to re-render a single XML file
def re_render_xml(filename, rotation=True):
    if filename.endswith(".xml"):
        image_name = filename.split(".")[0]
        # Generate image 
        # Construct the command
        command = ""
        command += f"cd {program_path} && ./main " 
        command += f"-h 1.0 -o {output_path} -name {image_name} -tile none -f {os.path.join(output_path, filename)}"
        if rotation:
            command += " -r"
        result = subprocess.run(command, shell=True, capture_output=True, text=True)
        return result

# Re-render XML files in parallel with progress bar
with concurrent.futures.ThreadPoolExecutor() as executor:
    xml_list = os.listdir(output_path)
    xml_list.sort()
    xml_list = [filename for filename in xml_list if filename.endswith(".xml")]
    # Check if the _0.jpeg, _120.jpeg, and _240.jpeg is already generated
    # Then skip the xml file
    filiterd_xml_list = []
    for xml_file in xml_list:
        data_name = xml_file.split('.')[0]
        jpeg_files = [f"{data_name}_{angle}.jpeg" for angle in [0, 120, 240]]

        # Check if the JPEG files already exist
        if all(os.path.exists(os.path.join(output_path, jpeg_file)) for jpeg_file in jpeg_files):
            print(f"Skipping {xml_file} as JPEG files already exist.")
            continue
        else:
            filiterd_xml_list.append(xml_file)
    xml_list = filiterd_xml_list
    futures = [executor.submit(re_render_xml, filename) for filename in xml_list if filename.endswith(".xml")]
    
    # Use tqdm to show progress bar
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Re-rendering XML files"):
        result = future.result()
        # Check if the command was successful
        if result.returncode != 0:
            print(result.stdout)  # Print the standard output
            print(result.stderr)  # Print the error output
            raise Exception("Command failed")

Re-rendering XML files: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40000/40000 [2:35:45<00:00,  4.28it/s]


In [5]:
# List jpg and xml files and move them to the dataset directory
for filename in os.listdir(output_path):
    if filename.endswith(".jpeg"):
        shutil.move(os.path.join(output_path, filename), images_path)
    elif filename.endswith(".xml"):
        shutil.move(os.path.join(output_path, filename), xml_path)

In [ ]:
# Test loading the generated dataset
from plant_dataset import PlantDataset 
# Show some images
import matplotlib.pyplot as plt
import numpy as np
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from plant_dataset import collate_fn
transform = transforms.Compose([
                        transforms.ToTensor(),
                        # transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])
train_dataset = PlantDataset("../data/generated_Nov14_20224", load_depth=False, preload=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)
import cv2
n = 5
for i in range(n):
    #image, vecs, _ = train_dataset[-i-1]
    #image, vecs, _ = train_dataset[i]
    image, vecs, _ = next(iter(train_loader))
    image = image.permute(1, 2, 0)
    image_rgb = image[:, :, :3]
    img = cv2.normalize(np.array(image_rgb.cpu()), None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    # img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(5, 2, 2*i+1)
    plt.imshow(img)
    plt.title(f"Image {i}")
    plt.axis('off')

    if train_dataset.load_depth:
        image_depth = image[:, :, 3]
        img = cv2.normalize(np.array(image_depth.cpu()), None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
        plt.subplot(5, 2, 2*i+2)
        plt.imshow(img, cmap='gray')
        plt.title("Depth")
        plt.axis('off')